# Lab 7 — Completed Exercises

**Student:** Abdulaziz Saqer BinRuqush  
**Student ID:** 2230007229

Instructor notebook: [Lab7_LLMs.ipynb](https://github.com/MohammedMosuily/NLP_Lab7)  
Source commit: `6f2496b6d64f1c9193772da0f05dab8f70deadb3`

Run the setup commands in the course README before running all cells.
Instructor examples and task instructions are retained below.


In [1]:
from pathlib import Path
import os
import sys
import random
import numpy as np
import pandas as pd
from IPython.display import display

LAB_DIR = Path.cwd()
if not (LAB_DIR / 'Lab7_LLMs.ipynb').is_file():
    LAB_DIR = LAB_DIR / "Lab7"
if not (LAB_DIR / 'Lab7_LLMs.ipynb').is_file():
    raise FileNotFoundError("Run from the course root or this lab's folder.")
PROJECT_ROOT = LAB_DIR.parent
os.environ.setdefault("KAGGLEHUB_CACHE", str(PROJECT_ROOT / ".cache" / "kagglehub"))
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache" / "huggingface"))
os.environ.setdefault("NLTK_DATA", str(PROJECT_ROOT / ".cache" / "nltk_data"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from course_data import dataset_file
import nltk
nltk.data.path.insert(0, os.environ["NLTK_DATA"])
random.seed(42)
np.random.seed(42)
print("Lab 7 | Abdulaziz Saqer BinRuqush | 2230007229")


Lab 7 | Abdulaziz Saqer BinRuqush | 2230007229


# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [2]:
import json
MODEL_REVISIONS = json.loads((PROJECT_ROOT / "model_revisions.json").read_text(encoding="utf-8"))
import torch
import transformers
torch.set_num_threads(2)
print("PyTorch:", torch.__version__, "Transformers:", transformers.__version__)


PyTorch: 2.8.0+cpu Transformers: 4.57.6


In [3]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [4]:
import gc
set_seed(42)
generator = pipeline("text-generation", model="distilgpt2", revision=MODEL_REVISIONS["distilgpt2"], device=-1)
prompt = "Natural language processing is"
output = generator(prompt, max_new_tokens=30, do_sample=False, temperature=1.0,
                   pad_token_id=generator.tokenizer.eos_token_id)
print(output[0]["generated_text"])
del generator
gc.collect()


Device set to use cpu


Natural language processing is a common feature of the language processing process.























40

# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small", revision=MODEL_REVISIONS["google/flan-t5-small"])
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", revision=MODEL_REVISIONS["google/flan-t5-small"])
model.eval()
prompt = "Translate to French: I love natural language processing."
with torch.no_grad():
    output = model.generate(**tokenizer(prompt, return_tensors="pt"), max_new_tokens=50, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))
del model, tokenizer, output
gc.collect()


Je s'agissant de la technologie de linguage naturelle.


117

# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [6]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", revision=MODEL_REVISIONS["distilbert-base-uncased-finetuned-sst-2-english"], device=-1)
text = "This NLP lab is easy and interesting."
result = classifier(text)
print(result)


Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [7]:
families = [
    ("Sentiment classification", "Encoder-only", "Reads the full input to assign a sentiment label."),
    ("Machine translation", "Encoder–decoder", "Encodes the source text and generates a target-language sequence."),
    ("Open-ended text generation", "Decoder-only", "Generates a continuation one token at a time."),
    ("Named Entity Recognition", "Encoder-only", "Uses contextual token representations to classify each token."),
    ("Text summarization", "Encoder–decoder", "Encodes the document and generates a shorter output sequence."),
]
display(pd.DataFrame(families, columns=["Task", "Model family", "Reason"]))


,Task,Model family,Reason
0,Sentiment classification,Encoder-only,Reads the full input to assign a sentiment label.
1,Machine translation,Encoder–decoder,Encodes the source text and generates a target...
2,Open-ended text generation,Decoder-only,Generates a continuation one token at a time.
3,Named Entity Recognition,Encoder-only,Uses contextual token representations to class...
4,Text summarization,Encoder–decoder,Encodes the document and generates a shorter o...


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [8]:
sentences = ["The movie was excellent.", "The lecture was difficult to understand.",
             "I enjoyed learning about transformers."]
predictions = classifier(sentences)
for sentence, prediction in zip(sentences, predictions):
    print(f"{sentence} → {prediction['label']} (score: {prediction['score']:.4f})")
assert len(predictions) == 3 and all(0 <= p["score"] <= 1 for p in predictions)
del classifier
gc.collect()


The movie was excellent. → POSITIVE (score: 0.9999)
The lecture was difficult to understand. → NEGATIVE (score: 0.9997)
I enjoyed learning about transformers. → POSITIVE (score: 0.9997)


0

## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [9]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small", revision=MODEL_REVISIONS["google/flan-t5-small"])
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", revision=MODEL_REVISIONS["google/flan-t5-small"])
model.eval()
paragraph = ("Natural language processing helps computers work with human language. "
             "Transformer models use attention to identify useful relationships between words. "
             "Encoder models support classification, encoder-decoder models support translation "
             "and summarization, and decoder models generate text one token at a time.")
inputs = tokenizer("Summarize the following paragraph in one sentence: " + paragraph, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    summary_tokens = model.generate(**inputs, max_new_tokens=60, do_sample=False, num_beams=4,
                                    no_repeat_ngram_size=3)
summary = tokenizer.decode(summary_tokens[0], skip_special_tokens=True)
print("Paragraph:", paragraph)
print("FLAN-T5 summary:", summary)
assert summary.strip()
del model, tokenizer, inputs, summary_tokens
gc.collect()


Paragraph: Natural language processing helps computers work with human language. Transformer models use attention to identify useful relationships between words. Encoder models support classification, encoder-decoder models support translation and summarization, and decoder models generate text one token at a time.
FLAN-T5 summary: Learn how to use natural language processing.


117


## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


## Task 5 — Reflection answer

A decoder-only model generates each next token from the conversation
tokens that came before it. This autoregressive process lets it produce
responses of different lengths while using the conversation as context.
Causal attention prevents the model from relying on future tokens during
training, matching how it generates a reply. Instruction tuning and
preference training can further teach the model to respond helpfully to
a user's requests.


# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [10]:
# Optional challenge: sampling without resetting the seed between calls.
generator = pipeline("text-generation", model="distilgpt2", revision=MODEL_REVISIONS["distilgpt2"], device=-1)
sampling_prompt = "Natural language processing can help people"
for run in range(3):
    sampled = generator(sampling_prompt, max_new_tokens=30, do_sample=True,
                        top_k=50, top_p=0.95, temperature=0.8,
                        pad_token_id=generator.tokenizer.eos_token_id)
    print(f"Run {run + 1}: {sampled[0]['generated_text']}")
del generator
gc.collect()


Device set to use cpu


Run 1: Natural language processing can help people with complex problems with basic English, or learning to use their language properly.

















Run 2: Natural language processing can help people navigate more complex and complex language-related topics.






















Run 3: Natural language processing can help people better understand the language of their choice.


0

Sampling chooses tokens randomly from a probability distribution rather
than always choosing its largest value. Without resetting the seed,
successive calls use different random draws, so the same prompt can
produce different continuations. Resetting the same seed before each
call would make the draws reproducible in the same environment.
